# MichiganCast Demo 02: Analysis and Baselines

This notebook demonstrates automated EDA, rare event metric selection, and classical baseline model evaluation.

## 0. Objectives and Conclusions

**Objectives**
1. Generate a repeatable EDA report from cleaned data.
2. Quantify class imbalance and seasonal behavior.
3. Train baseline models under a shared evaluation protocol.
4. Compare model quality with PR AUC oriented metrics.

**Conclusion Template**
- Which data patterns explain baseline performance.
- Which baseline model offers the strongest precision recall tradeoff.
- Which failure modes remain before neural modeling.

## 1. Inputs and Outputs

| Type | Path |
|---|---|
| EDA module | `src/analysis/eda_report.py` |
| Baseline module | `src/models/baselines.py` |
| Evaluation metrics | `src/eval/metrics.py` |
| Input dataset | `data/interim/traverse_city_daytime_clean_v1.csv` |
| EDA summary | `artifacts/reports/eda_summary.json` |
| Baseline report | `artifacts/reports/baseline_results.json` |
| Figures | `artifacts/figures/*` |

In [ ]:
from pathlib import Path
import subprocess
import json
import pandas as pd

CLEAN_CSV = 'data/interim/traverse_city_daytime_clean_v1.csv'
EDA_SUMMARY = 'artifacts/reports/eda_summary.json'
BASELINE_REPORT = 'artifacts/reports/baseline_results.json'
FIG_DIR = 'artifacts/figures'

print('clean csv exists:', Path(CLEAN_CSV).exists())
print('eda summary exists:', Path(EDA_SUMMARY).exists())
print('baseline report exists:', Path(BASELINE_REPORT).exists())

## 2. Method and Implementation Using src Modules

Commands are printed by default. Set `DEMO_EXECUTE=True` to run and regenerate artifacts.

In [ ]:
DEMO_EXECUTE = False

def run_or_print(cmd: str):
    print('\n$ ' + cmd)
    if not DEMO_EXECUTE:
        print('[skip] DEMO_EXECUTE=False, only showing command.')
        return None
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
    return proc.returncode

### 2.1 Generate Automated EDA Report T13

**Purpose** Produce objective statistics and figures for class distribution, seasonality, correlation, and temporal drift.

In [ ]:
cmd_eda = (
    'scripts/run_in_pytorch_env.sh python -m src.analysis.eda_report '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--summary artifacts/reports/eda_summary.json '
    '--fig-dir artifacts/figures'
)
run_or_print(cmd_eda)

### 2.2 Train Classical Baselines T14 and T15

**Purpose** Establish logistic regression and tree based baselines using a unified metric stack.

In [ ]:
cmd_baseline = (
    'scripts/run_in_pytorch_env.sh python -m src.models.baselines '
    '--input data/interim/traverse_city_daytime_clean_v1.csv '
    '--report artifacts/reports/baseline_results.json '
    '--fig-dir artifacts/figures'
)
run_or_print(cmd_baseline)

## 3. Results and Interpretation

Focus on:
1. Positive class ratio and imbalance severity.
2. PR AUC and recall behavior under precision constraints.
3. Calibration quality represented by Brier score.

In [ ]:
if Path(EDA_SUMMARY).exists():
    eda_payload = json.loads(Path(EDA_SUMMARY).read_text(encoding='utf-8'))
    class_summary = eda_payload.get('class_summary', {})
    print('rows_analyzed:', eda_payload.get('rows_analyzed'))
    print('positive_rate:', class_summary.get('positive_rate'))
    print('imbalance_conclusion:', class_summary.get('class_imbalance_conclusion'))
    print('\nEDA figures:')
    for k, v in eda_payload.get('figure_paths', {}).items():
        print(f'- {k}: {v}')
else:
    print('EDA summary not found. Run section 2.1 first.')

In [ ]:
if Path(BASELINE_REPORT).exists():
    baseline_payload = json.loads(Path(BASELINE_REPORT).read_text(encoding='utf-8'))
    rows = []
    for model_name, detail in baseline_payload.get('models', {}).items():
        val = detail.get('validation', {})
        rows.append({
            'model': model_name,
            'val_pr_auc': val.get('pr_auc'),
            'val_f1': val.get('f1'),
            'val_recall': val.get('recall'),
            'val_precision': val.get('precision'),
            'val_recall_at_precision': val.get('recall_at_precision'),
            'val_brier': val.get('brier'),
        })
    df_baseline = pd.DataFrame(rows).sort_values('val_pr_auc', ascending=False)
    display(df_baseline)
else:
    print('Baseline report not found. Run section 2.2 first.')

### 3.1 Presentation Highlights

1. Explain why PR AUC is prioritized over accuracy.
2. Show confusion matrices to expose false negative risk.
3. Use baseline gaps to motivate multimodal modeling.

## 4. Engineering Notes and Next Steps

- Add confidence intervals through repeated time slice evaluation.
- Introduce feature stability checks across yearly splits.
- Track baseline drift in a periodic monitoring job.

## Appendix. Reproducible Commands

```bash
scripts/run_in_pytorch_env.sh python -m src.analysis.eda_report --input data/interim/traverse_city_daytime_clean_v1.csv --summary artifacts/reports/eda_summary.json --fig-dir artifacts/figures
scripts/run_in_pytorch_env.sh python -m src.models.baselines --input data/interim/traverse_city_daytime_clean_v1.csv --report artifacts/reports/baseline_results.json --fig-dir artifacts/figures
```